In [ ]:
import numpy as np
from numpy.linalg import eigh
import torch

**A note on the code below:**

When we conduct the diagonalization of $V\Lambda V^*$, we wish to use `eigh` instead of `eig` since `eigh` uses a faster algorithm that **assumes the matrix is Hermitian** $(A = A^*)$. That is to say, we should only pass in Hermitian matrices into `eigh`. This poses a problem since our normal matrix $V\Lambda V^*$ is not Hermitian. Below is what $V\Lambda V^*$ is when $P=3$:

\begin{bmatrix}
-0.5 & 0.866 & 1.118 \\
-0.866 & -0.5 & 1.936 \\
-1.118 & -1.936 & -0.5
\end{bmatrix}

However, $K = V\Lambda V^* - \frac{1}{2}I$ is *skew-Hermitian* $(-A = A^*)$. This means $V\Lambda V^*$ is of the form:
$$V\Lambda V^* = \alpha I + K$$
Where $\alpha\in\mathbb{R}$ and $K$ is skew-Hermitian. Now, observe that:
$$V\Lambda V^*v = \lambda v \iff Kv = (\lambda - \alpha)v$$
That is to say that $V\Lambda V^*$ and its skew part $K$ **share the same eigenvectors**, albeit their eigenvalues are shifted by $\alpha$.

If we multiply both sides by $-i$:
$$-iV\Lambda V^* = -i\alpha I + -iK$$
We then have:
$$(-iV\Lambda V^*)v = \mu v \iff (-iK)v = (\mu + i\alpha)v$$
Which is to say the **eigenvectors of $V\Lambda V^*$, $K$, $-iV\Lambda V^*$, and $-iK$ are all the same!** Their eigenvalues are simply scaled $(\mu = -i\lambda)$. Lucky for us, $-iK$ is Hermitian, so we will seek to (indirectly) use it as the input for `eigh`.

To see why $-iK$ is Hermitian, first note that it is a fact that multiplying a skew-Hermitian matrix by $-i$ yields a Hermitian matrix:
$$\text{Sketch proof: } -K=K^* \implies -iK = i(-K) = iK^* = (-iK)^*$$

<!-- So we have the following:
$$-iV\Lambda V^* = \underbrace{-i\alpha I}_{\text{skew-Hermitian}} + \underbrace{-iK}_{\text{Hermitian}} $$ -->

What `eigh` really does is "take the lower triangular part of the matrix and assumes that the upper triangular part is defined by the symmetry of the matrix" [(stackoverflow)](https://stackoverflow.com/questions/45434989/numpy-difference-between-linalg-eig-and-linalg-eigh). The [documentation](https://numpy.org/doc/2.3/reference/generated/numpy.linalg.eigh.html) for `eigh` actually states in regards to the optional second argument `UPLO`:
> Specifies whether the calculation is done with the lower triangular part of a (‘L’, default) or the upper triangular part (‘U’). Irrespective of this value only the real parts of the diagonal will be considered in the computation to preserve the notion of a Hermitian matrix. It therefore follows that the imaginary part of the diagonal will always be treated as zero.

Here we note that $-iV\Lambda V^*$ has a *purely imaginary diagonal*, so if we pass it into `eigh`, it will effectively be treated as $-iK$ since $K$ is just $V\Lambda V^*$ with the diagonal removed. So we leverage this technical nuance to avoid computing $K$ and just pass in `VΛV * -1j` to yield the desired eigenvector matrix $V$.




In [ ]:
def DPLR_HiPPO(P):
    
    # HiPPO
    P_arr = np.arange(P)
    M = np.sqrt(2*P_arr + 1) # creates a P-dimensional array according to HiPPO-LegS
    A = -np.tril(np.outer(M,M)) + np.diag(P_arr) # A_nk in Appendix C.1 of S4 paper
    
    # DPLR
    Q = np.sqrt(P_arr + 0.5) # low-rank array (i.e. vector) part of the "low rank correction". See "adding..." in C.1 of S4 paper
    VΛV = A + np.outer(Q,Q)
    Λ_real = np.diagonal(VΛV) # this is just a P-dimensional array of -0.5's
    Λ_imag, V = eigh(VΛV * -1j)
    Q_tilde = V.conj() @ Q
    B_tilde = V.conj() @ B   
    
    return (
        # note the imaginary eigenvalues are being multiplied by i to account for the -i scaling above
        # recall that \mu = -i\lambda, so to recover \lambda we just multiply by i
        torch.tensor(np.asarray(Λ_real + 1j * Λ_imag), dtype=th.complex64),
        torch.tensor(np.asarray(Q_tilde)),
        torch.tensor(np.asarray(B_tilde)),
        torch.tensor(np.asarray(V), dtype=th.complex64),
        torch.tensor(np.asarray(B)),
    ) 
    

If you wish to have confirmation that this diagonalization works, run the cell below for any value of `P` you desire.

In [ ]:
P = 3
P_arr = np.arange(P)
M = np.sqrt(2*P_arr + 1) # creates a P-dimensional array according to HiPPO-LegS
A = -np.tril(np.outer(M,M)) + np.diag(P_arr) # A_nk in Appendix C.1 of S4 paper

# NPLR
Q = np.sqrt(P_arr + 0.5) # part of the "low rank correction".
VΛV = A + np.outer(Q,Q)
Λ_real = np.diagonal(VΛV) # this is just a P-dimensional array of -0.5's
Λ_imag, V = eigh(VΛV * -1j)

Λ = np.diag(Λ_real + 1j*Λ_imag)
print(np.allclose(A,V @ Λ @ V.conj().T - np.outer(Q,Q), atol=1e-4, rtol=1e-4))
